# Notebook 08 — Business Application and Economic Interpretation

Notebook 07 selected the full eight-block log-response specification estimated using refined weighted least squares (WLS) as the final model.

With model development and validation complete, the validated specification is now re-estimated using the full analysis sample to obtain the final coefficient estimates.

This notebook focuses on economic interpretation and practical application. The final model is used to quantify systematic fare differences associated with itinerary, purchase, carrier, and geographic characteristics and to develop a fare-benchmarking framework.

No further variable selection or model refinement is conducted.

## 1. Prepare the Full Analysis Sample

The full cleaned sample of 90,086 observations is used for final estimation.

To preserve the validated model specification, the carrier and origin-state groupings developed from the original model-building sample are reproduced and then applied unchanged to the full dataset.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

import statsmodels.formula.api as smf

In [2]:
PROJECT_DIR = Path.cwd().resolve()

if PROJECT_DIR.name == "notebooks":
    PROJECT_DIR = PROJECT_DIR.parent

PROCESSED_DATA_DIR = PROJECT_DIR / "data" / "processed"
OUTPUT_DIR = PROJECT_DIR / "outputs"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DATA_FILE = (
    PROCESSED_DATA_DIR
    / "db1c_ticket_prepared_202606.parquet"
)

df = pd.read_parquet(DATA_FILE)

print(f"Rows:    {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")

Rows:    90,086
Columns: 16


### 1.1 Reproduce the Validated Category Groupings

Carrier and origin-state categories were consolidated using the model-building sample during model development.

The same category definitions are retained here so that final estimation uses exactly the specification that was validated in Notebook 07.

In [3]:
SPLIT_SEED = 42
TRAIN_SHARE = 0.50
MIN_CATEGORY_COUNT = 100

rng = np.random.default_rng(SPLIT_SEED)

random_order = rng.permutation(len(df))
split_point = int(TRAIN_SHARE * len(df))

train_idx = random_order[:split_point]

model_df = (
    df.iloc[train_idx]
    .copy()
    .reset_index(drop=True)
)

In [4]:
carrier_counts = model_df[
    "RpCarrier"
].value_counts()

major_carriers = carrier_counts[
    carrier_counts >= MIN_CATEGORY_COUNT
].index

state_counts = model_df[
    "OriginState"
].value_counts()

major_states = state_counts[
    state_counts >= MIN_CATEGORY_COUNT
].index

In [5]:
full_df = df.copy()

full_df["RpCarrier_Model"] = (
    full_df["RpCarrier"].where(
        full_df["RpCarrier"].isin(
            major_carriers
        ),
        "Other"
    )
)

full_df["OriginState_Model"] = (
    full_df["OriginState"].where(
        full_df["OriginState"].isin(
            major_states
        ),
        "Other"
    )
)

full_df["LogTotalAmount"] = np.log(
    full_df["TotalAmount"]
)

print(
    f"Carrier model categories: "
    f"{full_df['RpCarrier_Model'].nunique()}"
)

print(
    f"Origin-state model categories: "
    f"{full_df['OriginState_Model'].nunique()}"
)

Carrier model categories: 12
Origin-state model categories: 43


#### Interpretation

The validated category structure is preserved when the full sample is reconstructed.

The additional observations are therefore used to improve final parameter estimation without changing the model specification selected and validated previously.

## 2. Estimate the Final Full-Sample Model

The validated eight-block log-response specification is re-estimated using all 90,086 observations.

The refined WLS variance-function form is held fixed, while its coefficients are re-estimated using the full sample. This allows all available information to contribute to the final coefficient and weight estimates without introducing a new model specification.

### 2.1 Fit the Full-Sample Log-OLS Model

The validated eight-block log-response specification is first estimated using OLS. The resulting fitted values and residuals are used to estimate the refined variance function required for WLS.

In [6]:
final_formula = """
LogTotalAmount ~
    TotalDistance
    + Coupons
    + C(RoundTrip, Treatment(reference=0))
    + C(OnLine, Treatment(reference=1))
    + C(PurchaseWindowGroup, Treatment(reference='21AP'))
    + C(RpCarrier_Model, Treatment(reference='AA'))
    + C(ItinGeoType, Treatment(reference=2))
    + C(OriginState_Model, Treatment(reference='CA'))
"""

full_log_model = smf.ols(
    formula=final_formula,
    data=full_df
).fit()

print(
    f"Observations: "
    f"{int(full_log_model.nobs):,}"
)

print(
    f"Parameters: "
    f"{int(full_log_model.df_model + 1)}"
)

Observations: 90,086
Parameters: 61


### 2.2 Estimate the Refined Variance Function

Absolute residuals from the full-sample log-OLS model are used to re-estimate the validated standard-deviation function.

The functional form remains unchanged from Notebook 06.

In [7]:
final_wls_data = full_df.copy()

final_wls_data["Fitted_LogFare"] = (
    full_log_model.fittedvalues.to_numpy()
)

final_wls_data["Abs_Residual"] = np.abs(
    full_log_model.resid.to_numpy()
)

final_sd_model = smf.ols(
    """
    Abs_Residual ~
        Fitted_LogFare
        + I(Fitted_LogFare**2)
        + TotalDistance
        + C(RoundTrip, Treatment(reference=0))
        + C(OnLine, Treatment(reference=1))
    """,
    data=final_wls_data
).fit()

final_sd = final_sd_model.predict(
    final_wls_data
)

final_wls_data["WLS_Weight"] = (
    1 / final_sd**2
)

print(
    f"Estimated SD range: "
    f"{final_sd.min():.6f} to "
    f"{final_sd.max():.6f}"
)

print(
    f"Max / min weight ratio: "
    f"{final_wls_data['WLS_Weight'].max() / final_wls_data['WLS_Weight'].min():.2f}"
)

Estimated SD range: 0.283378 to 1.406863
Max / min weight ratio: 24.65


### 2.3 Final Refined WLS Model

In [8]:
final_wls_model = smf.wls(
    formula=final_formula,
    data=final_wls_data,
    weights=final_wls_data[
        "WLS_Weight"
    ]
).fit()

print(
    f"Observations: "
    f"{int(final_wls_model.nobs):,}"
)

print(
    f"Parameters: "
    f"{int(final_wls_model.df_model + 1)}"
)

Observations: 90,086
Parameters: 61


In [9]:
final_coefficients = pd.DataFrame({
    "Coefficient":
        final_wls_model.params.index,
    "Estimate":
        final_wls_model.params.values,
    "Standard_Error":
        final_wls_model.bse.values,
    "p_value":
        final_wls_model.pvalues.values
})

display(
    final_coefficients.head(15)
)

,Coefficient,Estimate,Standard_Error,p_value
0,Intercept,5.601052,0.006489,0.000000e+00
1,"C(RoundTrip, Treatment(reference=0))[T.1]",0.439569,0.004007,0.000000e+00
2,"C(OnLine, Treatment(reference=1))[T.0]",0.098967,0.005460,2.690459e-73
3,"C(PurchaseWindowGroup, Treatment(reference='21...",-0.308145,0.003183,0.000000e+00
4,"C(PurchaseWindowGroup, Treatment(reference='21...",-0.317335,0.005040,0.000000e+00
5,"C(RpCarrier_Model, Treatment(reference='AA'))[...",-0.121701,0.007342,1.283825e-61
6,"C(RpCarrier_Model, Treatment(reference='AA'))[...",-0.275193,0.008301,1.473574e-239
7,"C(RpCarrier_Model, Treatment(reference='AA'))[...",0.054283,0.004996,1.762770e-27
8,"C(RpCarrier_Model, Treatment(reference='AA'))[...",-0.746946,0.007799,0.000000e+00
9,"C(RpCarrier_Model, Treatment(reference='AA'))[...",-1.292363,0.009988,0.000000e+00


#### Interpretation

The validated eight-block specification is successfully re-estimated using all 90,086 observations.

The estimated standard deviations remain positive across the full sample. The resulting WLS weights are somewhat more dispersed than in the model-building sample but remain finite and usable across all observations.

The full-sample coefficient estimates are therefore used for the final economic interpretation and fare-benchmarking analysis.

In [10]:
final_coefficients.to_csv(
    OUTPUT_DIR
    / "08_final_model_coefficients.csv",
    index=False
)

## 3. Economic Interpretation of Fare Associations

Because the dependent variable is the logarithm of `TotalAmount`, coefficient interpretation depends on the type of predictor.

For a dummy or categorical coefficient $\beta$, the exact percentage difference relative to the reference category is

$$
100\left(e^\beta-1\right)\%.
$$

For a continuous predictor, the same transformation can be applied to a meaningful change in the predictor:

$$
100\left(e^{\beta\Delta x}-1\right)\%.
$$

The estimated relationships are interpreted as conditional associations rather than causal effects.

### 3.1 Main Itinerary and Purchase Effects

The principal itinerary and purchase characteristics are translated into percentage fare differences while holding the remaining model variables fixed.

Continuous effects are reported over economically meaningful increments rather than per unit where appropriate.

In [11]:
def percent_effect(beta, change=1):
    return (
        np.exp(beta * change) - 1
    ) * 100

In [12]:
distance_beta = final_wls_model.params[
    "TotalDistance"
]

coupons_beta = final_wls_model.params[
    "Coupons"
]

roundtrip_name = (
    "C(RoundTrip, Treatment(reference=0))[T.1]"
)

online_name = (
    "C(OnLine, Treatment(reference=1))[T.0]"
)

main_effects = pd.DataFrame({
    "Effect": [
        "TotalDistance: +100 miles",
        "TotalDistance: +500 miles",
        "Coupons: +1 coupon",
        "Round trip vs one way",
        "OnLine = 0 vs OnLine = 1"
    ],
    "Percent_Difference": [
        percent_effect(
            distance_beta,
            change=100
        ),
        percent_effect(
            distance_beta,
            change=500
        ),
        percent_effect(
            coupons_beta
        ),
        percent_effect(
            final_wls_model.params[
                roundtrip_name
            ]
        ),
        percent_effect(
            final_wls_model.params[
                online_name
            ]
        )
    ]
})

display(
    main_effects.round(2)
)

,Effect,Percent_Difference
0,TotalDistance: +100 miles,1.58
1,TotalDistance: +500 miles,8.14
2,Coupons: +1 coupon,1.32
3,Round trip vs one way,55.20
4,OnLine = 0 vs OnLine = 1,10.40


In [13]:
purchase_effects = []

for coefficient, estimate in (
    final_wls_model.params.items()
):
    if "PurchaseWindowGroup" in coefficient:

        category = (
            coefficient
            .split("[T.")[-1]
            .rstrip("]")
        )

        purchase_effects.append({
            "PurchaseWindowGroup": category,
            "Reference": "21AP",
            "Estimate": estimate,
            "Percent_Difference":
                percent_effect(estimate)
        })

purchase_effects = pd.DataFrame(
    purchase_effects
)

display(
    purchase_effects.round(2)
)

,PurchaseWindowGroup,Reference,Estimate,Percent_Difference
0,2290,21AP,-0.31,-26.52
1,91UP,21AP,-0.32,-27.19


#### Interpretation

Several itinerary and purchase characteristics show economically meaningful associations with ticket fares, holding the remaining model variables fixed.

An additional 100 miles of itinerary distance is associated with approximately a 1.58% higher fare, while a 500-mile increase corresponds to approximately 8.14%. Each additional coupon, representing an additional ticketed itinerary segment, is associated with approximately a 1.32% higher fare.

Round-trip itineraries are associated with approximately 55.20% higher total ticket amounts than one-way itineraries. This relatively large difference should be interpreted in the context of the dependent variable being the total amount paid for the entire ticket rather than as a causal round-trip premium.

Itineraries with `OnLine = 0` are associated with fares approximately 10.40% higher than single-carrier itineraries (`OnLine = 1`).

Relative to the `21AP` purchase-window reference category, fares are approximately 26.52% lower for `2290` and 27.19% lower for `91UP`, conditional on the other observed characteristics.

These estimates describe conditional associations rather than causal effects.

### 3.2 Carrier Effects

Carrier coefficients measure the conditional fare difference relative to American Airlines (`AA`), holding itinerary, purchase, and geographic characteristics fixed.

The coefficients are converted into percentage differences to provide a more direct economic interpretation.

In [14]:
carrier_effects = []

for coefficient, estimate in final_wls_model.params.items():

    if "RpCarrier_Model" in coefficient:

        carrier = (
            coefficient
            .split("[T.")[-1]
            .rstrip("]")
        )

        carrier_effects.append({
            "Carrier": carrier,
            "Reference": "AA",
            "Estimate": estimate,
            "Percent_Difference":
                percent_effect(estimate),
            "p_value":
                final_wls_model.pvalues[
                    coefficient
                ]
        })

carrier_effects = pd.DataFrame(
    carrier_effects
).sort_values(
    "Percent_Difference",
    ascending=False
)

display(
    carrier_effects.round({
        "Estimate": 4,
        "Percent_Difference": 2,
        "p_value": 4
    })
)

,Carrier,Reference,Estimate,Percent_Difference,p_value
2,DL,AA,0.0543,5.58,0.0
8,UA,AA,0.0236,2.39,0.0
9,WN,AA,-0.0844,-8.09,0.0
0,AS,AA,-0.1217,-11.46,0.0
1,B6,AA,-0.2752,-24.06,0.0
6,Other,AA,-0.4076,-33.47,0.0
7,SY,AA,-0.5979,-45.01,0.0
5,MX,AA,-0.7127,-50.97,0.0
3,F9,AA,-0.7469,-52.62,0.0
10,XP,AA,-0.9874,-62.75,0.0


#### Interpretation

Carrier identity is strongly associated with ticket fares after controlling for itinerary distance, complexity, trip structure, purchase window, and geography.

Relative to American Airlines (`AA`), Delta (`DL`) and United (`UA`) are associated with approximately 5.58% and 2.39% higher fares, respectively. Southwest (`WN`) and Alaska (`AS`) are associated with fares approximately 8.09% and 11.46% lower than `AA`.

Larger negative differences are observed for several other carriers. For example, `B6` is associated with fares approximately 24.06% lower than `AA`, while `F9`, `XP`, and `G4` are associated with substantially lower conditional fares.

All reported carrier effects are statistically significant at conventional levels. The displayed p-values of 0.0 reflect rounding rather than exact zero values.

These estimates should be interpreted as conditional fare associations rather than causal carrier effects. Differences may also reflect unobserved route, service, fare-class, or competitive characteristics not fully captured by the model. The `Other` category represents a pooled group of low-frequency carriers and should not be interpreted as a single airline.

### 3.3 Geographic Effects

The final model contains two geographic components:

- `ItinGeoType`, which distinguishes the itinerary's broad geographic classification;
- `OriginState_Model`, which captures origin-state differences relative to California (`CA`).

These coefficients measure conditional fare differences after controlling for the remaining itinerary, purchase, and carrier characteristics.

In [15]:
geo_type_name = (
    "C(ItinGeoType, Treatment(reference=2))[T.1]"
)

geo_type_effect = pd.DataFrame({
    "Comparison": [
        "ItinGeoType 1 vs 2"
    ],
    "Estimate": [
        final_wls_model.params[
            geo_type_name
        ]
    ],
    "Percent_Difference": [
        percent_effect(
            final_wls_model.params[
                geo_type_name
            ]
        )
    ],
    "p_value": [
        final_wls_model.pvalues[
            geo_type_name
        ]
    ]
})

display(
    geo_type_effect.round({
        "Estimate": 4,
        "Percent_Difference": 2,
        "p_value": 4
    })
)

,Comparison,Estimate,Percent_Difference,p_value
0,ItinGeoType 1 vs 2,0.0694,7.19,0.0


In [16]:
state_effects = []

for coefficient, estimate in (
    final_wls_model.params.items()
):

    if "OriginState_Model" in coefficient:

        state = (
            coefficient
            .split("[T.")[-1]
            .rstrip("]")
        )

        state_effects.append({
            "OriginState": state,
            "Reference": "CA",
            "Estimate": estimate,
            "Percent_Difference":
                percent_effect(estimate),
            "p_value":
                final_wls_model.pvalues[
                    coefficient
                ]
        })

state_effects = pd.DataFrame(
    state_effects
).sort_values(
    "Percent_Difference",
    ascending=False
)

display(
    state_effects.round({
        "Estimate": 4,
        "Percent_Difference": 2,
        "p_value": 4
    })
)

,OriginState,Reference,Estimate,Percent_Difference,p_value
1,AL,CA,0.1615,17.52,0.0000
0,AK,CA,0.1441,15.50,0.0000
19,MN,CA,0.1423,15.30,0.0000
29,OK,CA,0.1313,14.04,0.0000
2,AR,CA,0.1309,13.98,0.0000
20,MO,CA,0.1301,13.90,0.0000
18,MI,CA,0.1209,12.85,0.0000
12,IN,CA,0.1184,12.57,0.0000
41,WI,CA,0.1180,12.52,0.0000
16,MD,CA,0.1090,11.52,0.0000


#### Interpretation

Geographic characteristics remain associated with meaningful fare differences after controlling for itinerary structure, purchase timing, and carrier effects.

`ItinGeoType = 1`, representing non-contiguous domestic itineraries, is associated with fares approximately 7.19% higher than contiguous domestic itineraries (`ItinGeoType = 2`), holding the other observed characteristics fixed.

Origin-state effects also vary materially relative to California. Several states, including Alabama, Alaska, Minnesota, Oklahoma, and Arkansas, are associated with conditional fares more than 13% higher than California. At the other end, Hawaii is associated with fares approximately 25.87% lower than California, while Puerto Rico, Illinois, and New York also show lower conditional fare levels.

Not all state effects are statistically distinguishable from California. For example, the estimates for New Jersey, Oregon, Washington, Arizona, Nevada, Idaho, and Maine are not statistically significant at the 5% level.

The state coefficients should therefore be interpreted as conditional geographic associations rather than intrinsic state-level price effects. They may also capture unobserved differences in route mix, airport structure, competition, or other market characteristics not fully represented in the model.

## 4. Fare-Benchmarking Framework

The final model can also be used to construct a benchmark fare for an itinerary with a specified set of observable characteristics.

For each observation, the model first generates a predicted log fare:

$$
\widehat{\log(\text{Fare}_i)}=X_i\hat{\beta}.
$$

Exponentiating this value produces a model-implied fare benchmark on the original dollar scale:

$$
\widehat{\text{Fare}}_i
=
\exp\left(
\widehat{\log(\text{Fare}_i)}
\right).
$$

Because the model is estimated on the log scale, this retransformation is interpreted as a typical conditional fare benchmark rather than an exact prediction of the conditional mean fare.

These benchmarks are descriptive in-sample fitted values from the final full-sample model. Out-of-sample model performance was assessed separately using the held-out validation sample in Notebook 07.

In [17]:
final_wls_data["Predicted_LogFare"] = (
    final_wls_model.predict(
        final_wls_data
    )
)

final_wls_data["Benchmark_Fare"] = np.exp(
    final_wls_data["Predicted_LogFare"]
)

final_wls_data["Fare_Gap"] = (
    final_wls_data["TotalAmount"]
    - final_wls_data["Benchmark_Fare"]
)

final_wls_data["Fare_Gap_Percent"] = (
    (
        final_wls_data["TotalAmount"]
        / final_wls_data["Benchmark_Fare"]
    ) - 1
) * 100

In [18]:
benchmark_summary = final_wls_data[
    [
        "TotalAmount",
        "Benchmark_Fare",
        "Fare_Gap",
        "Fare_Gap_Percent"
    ]
].describe()

display(
    benchmark_summary.round(2)
)

,TotalAmount,Benchmark_Fare,Fare_Gap,Fare_Gap_Percent
count,90086.00,90086.00,90086.00,90086.00
mean,460.01,421.31,38.69,10.64
std,354.02,232.95,282.80,60.30
min,1.00,55.69,-10803.96,-99.56
25%,242.20,267.05,-80.87,-23.44
50%,388.40,378.17,-0.84,-0.26
75%,580.30,540.25,106.14,29.94
max,7935.82,14284.47,6500.02,2476.58


#### Interpretation

The model-implied benchmark fare is broadly centered on the observed ticket distribution.

The median observed fare is 388.40, compared with a median benchmark fare of 378.17. The median fare gap is close to zero at -0.84, indicating that the benchmark is well centered for a typical observation.

The mean observed fare exceeds the mean benchmark fare by approximately 38.69. This difference is expected because the fare distribution remains right-skewed and the benchmark is obtained by exponentiating predicted log fares, which represents a typical conditional fare rather than the arithmetic conditional mean.

The dispersion of fare gaps is substantial. The middle 50% of observations lie approximately between 23.44% below and 29.94% above their model-implied benchmark, reflecting meaningful residual variation not explained by the observable characteristics included in the model.

Extreme percentage gaps should be interpreted cautiously because very small or unusually large observed fares can generate large relative deviations.

## 5. Illustrative Fare-Benchmarking Examples

To illustrate the practical use of the model, representative tickets are selected from different parts of the fare-gap distribution.

For illustration, tickets within ±25% of the model-implied benchmark are classified as near the benchmark, while observations more than 25% below or above the benchmark are classified as relatively low or relatively high, respectively. These thresholds are descriptive and are not formal statistical cutoffs.

A positive fare gap indicates that the observed fare is higher than the model-implied benchmark for tickets with similar observable characteristics, while a negative gap indicates a relatively lower observed fare.

In [19]:
benchmark_examples = final_wls_data.copy()

benchmark_examples["Relative_Fare"] = np.select(
    [
        benchmark_examples["Fare_Gap_Percent"] <= -25,
        benchmark_examples["Fare_Gap_Percent"] >= 25
    ],
    [
        "Relatively low",
        "Relatively high"
    ],
    default="Near benchmark"
)

benchmark_examples[
    "Relative_Fare"
].value_counts()

Relative_Fare
Near benchmark     43582
Relatively high    25435
Relatively low     21069
Name: count, dtype: int64

In [20]:
targets = {
    "Relatively low": -30,
    "Near benchmark": 0,
    "Relatively high": 30
}

example_rows = []

for label, target in targets.items():

    subset = benchmark_examples[
        benchmark_examples["Relative_Fare"] == label
    ].copy()

    subset["Distance_From_Target"] = (
        subset["Fare_Gap_Percent"] - target
    ).abs()

    example_rows.append(
        subset
        .sort_values("Distance_From_Target")
        .iloc[0]
    )

fare_examples = pd.DataFrame(
    example_rows
)

fare_examples_display = (
    fare_examples[
        [
            "Relative_Fare",
            "TotalAmount",
            "Benchmark_Fare",
            "Fare_Gap_Percent",
            "TotalDistance",
            "Coupons",
            "RoundTrip",
            "OnLine",
            "PurchaseWindowGroup",
            "RpCarrier_Model",
            "ItinGeoType",
            "OriginState_Model"
        ]
    ]
    .reset_index(drop=True)
)

display(
    fare_examples_display.round(2)
)

,Relative_Fare,TotalAmount,Benchmark_Fare,Fare_Gap_Percent,TotalDistance,Coupons,RoundTrip,OnLine,PurchaseWindowGroup,RpCarrier_Model,ItinGeoType,OriginState_Model
0,Relatively low,142.99,204.27,-30.0,1276.0,2,1,1,2290,MX,2,NC
1,Near benchmark,322.40,322.40,0.0,414.0,1,0,1,21AP,AA,2,PA
2,Relatively high,631.80,486.00,30.0,3540.0,2,1,1,2290,AS,2,WA


#### Interpretation

The benchmarking framework identifies meaningful differences between observed fares and the model-implied benchmark after controlling for observable ticket characteristics.

Approximately 48% of observations fall within 25% of their benchmark fare, while about 23% are at least 25% below the benchmark and 28% are at least 25% above it. These bands are used only for illustration and should not be interpreted as formal statistical thresholds.

The representative examples show how the framework can distinguish relatively low-priced, benchmark-consistent, and relatively high-priced tickets with different itinerary characteristics. For example, the relatively low-priced ticket is approximately 30% below its model-implied benchmark, while the relatively high-priced example is approximately 30% above its benchmark.

The benchmark should therefore be interpreted as a conditional reference point rather than a prediction that every observed fare should closely match.

## 6. Business Application Summary

The final model provides two complementary outputs.

First, the coefficient estimates identify observable ticket characteristics associated with systematic fare differences. Distance, itinerary complexity, round-trip status, carrier structure, purchase timing, carrier identity, and geography all contribute to explaining cross-sectional fare variation.

Second, the fitted model provides a practical fare benchmark. For a ticket with specified observable characteristics, the model generates a typical conditional fare against which the observed ticket amount can be compared.

This framework can be used to identify tickets that appear relatively expensive or inexpensive after controlling for major observable fare characteristics. However, the benchmark does not incorporate all determinants of airline pricing. Unobserved factors such as fare class, real-time seat availability, route competition, loyalty status, and detailed ticket restrictions may contribute to the remaining variation.

Accordingly, the model is best viewed as an interpretable benchmarking tool rather than a complete airline pricing model.

In [21]:
main_effects.to_csv(
    OUTPUT_DIR / "08_main_effects.csv",
    index=False
)

purchase_effects.to_csv(
    OUTPUT_DIR / "08_purchase_window_effects.csv",
    index=False
)

carrier_effects.to_csv(
    OUTPUT_DIR / "08_carrier_effects.csv",
    index=False
)

state_effects.to_csv(
    OUTPUT_DIR / "08_origin_state_effects.csv",
    index=False
)

fare_examples_display.to_csv(
    OUTPUT_DIR / "08_fare_benchmark_examples.csv",
    index=False
)